In [43]:
!pip install -qU datasets pandas

In [44]:
import pandas as pd
from datasets import load_dataset

REPO = "bakrianoo/jabarti-llm-dataset"

pretrain = load_dataset(
    REPO,
    "pretrain",
    split="phase1_train"
).to_pandas()

finetune = load_dataset(
    REPO,
    "finetune",
    split="train"
).to_pandas()

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

In [45]:
print("Pretrain Schema")
print(pretrain.dtypes)

print("\nFinetune Schema")
print(finetune.dtypes)

Pretrain Schema
article_id              str
title                   str
url                     str
language                str
category                str
source_type             str
egypt_relevance         str
content_quality         str
topic_domain            str
importance              str
is_person              bool
is_stub_like           bool
ortho_convention        str
pageviews_30d         int64
chunk_strategy          str
chunk_index           int64
chunk_total           int64
pair_id                 str
dataset_version         str
ds_source               str
text                    str
section_index         int64
section_total         int64
ortho_stripped         bool
page_id                 str
record_id               str
sample_weight       float64
dtype: object

Finetune Schema
difficulty           str
type                 str
qa_id                str
dataset_version      str
question             str
article_title        str
answer               str
source_article_id    s

In [46]:
splits = {
    "pretrain / phase1_train": ("pretrain", "phase1_train"),
    "pretrain / phase1_eval": ("pretrain", "phase1_eval"),
    "pretrain / phase2_train": ("pretrain", "phase2_train"),
    "pretrain / phase2_eval": ("pretrain", "phase2_eval"),

    "finetune / train": ("finetune", "train"),
    "finetune / eval": ("finetune", "eval"),
}


CHARS_PER_TOKEN = 3.5

CONTEXT_LENGTHS = [512, 1024, 2048]


for label_name, (subset, split_name) in splits.items():

    df = load_dataset(
        REPO,
        subset,
        split=split_name
    ).to_pandas()


    print(f"\n===== {label_name} =====")

    print("ROWS:", df.shape[0])


    # ===== Check Nulls =====

    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]

    print("NULL VALUES:")
    print(
        "None"
        if nulls.empty
        else nulls.to_string()
    )


    # ===== Language Distribution =====

    lang_counts = df["language"].value_counts()
    total_rows = df.shape[0]

    for lang, n in lang_counts.items():

        print(
            f"{lang} => {n} samples",
            f"{n / total_rows * 100:.1f}%"
        )


    # ===== Text Length Distribution =====

    text_len_dist = 0

    if subset == "pretrain":

        text_len_dist = (
            df["text"]
            .astype(str)
            .str.len()
            .describe()
            .round(1)
        )


    elif subset == "finetune":

        text_len_dist = (
            pd.concat(
                [
                    df["question"].astype(str).str.len(),
                    df["answer"].astype(str).str.len()
                ],
                ignore_index=True
            )
            .describe()
            .round(1)
        )


    print("\n===== Text Length Distribution =====")
    print(text_len_dist)


    # ===== Approximate Token Length Per Model Sample =====

    if subset == "pretrain":

        sample_characters = (
            df["text"]
            .astype(str)
            .str.len()
        )


    elif subset == "finetune":

        sample_characters = (
            df["question"].astype(str).str.len()
            +
            df["answer"].astype(str).str.len()
        )


    approx_tokens = (
        sample_characters / CHARS_PER_TOKEN
    ).astype(int)


    print("\n===== Approximate Token Length =====")

    print(
        approx_tokens
        .describe()
        .round(1)
    )


    # ===== Context Window Analysis =====

    print("\n===== Context Window Analysis =====")

    for context_length in CONTEXT_LENGTHS:

        drop_percentage = (
            approx_tokens > context_length
        ).mean() * 100

        keep_percentage = 100 - drop_percentage

        print(
            f"Context Length: {context_length} | "
            f"Keep: {keep_percentage:.1f}% | "
            f"Drop: {drop_percentage:.1f}%"
        )

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]


===== pretrain / phase1_train =====
ROWS: 175476
NULL VALUES:
None
en => 125766 samples 71.7%
ar => 49710 samples 28.3%

===== Text Length Distribution =====
count    175476.0
mean       1553.0
std         702.1
min         178.0
25%        1169.0
50%        1653.0
75%        1913.0
max       20049.0
Name: text, dtype: float64

===== Approximate Token Length =====
count    175476.0
mean        443.3
std         200.6
min          50.0
25%         334.0
50%         472.0
75%         546.0
max        5728.0
Name: text, dtype: float64

===== Context Window Analysis =====
Context Length: 512 | Keep: 61.9% | Drop: 38.1%
Context Length: 1024 | Keep: 99.3% | Drop: 0.7%
Context Length: 2048 | Keep: 99.8% | Drop: 0.2%


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

...


===== pretrain / phase2_eval =====
ROWS: 2001
NULL VALUES:
None
ar => 1072 samples 53.6%
en => 929 samples 46.4%

===== Text Length Distribution =====
count     2001.0
mean      1413.0
std        742.8
min        279.0
25%        989.0
50%       1402.0
75%       1817.0
max      15024.0
Name: text, dtype: float64

===== Approximate Token Length =====
count    2001.0
mean      403.3
std       212.2
min        79.0
25%       282.0
50%       400.0
75%       519.0
max      4292.0
Name: text, dtype: float64

===== Context Window Analysis =====
Context Length: 512 | Keep: 73.6% | Drop: 26.4%
Context Length: 1024 | Keep: 99.3% | Drop: 0.7%
Context Length: 2048 | Keep: 99.8% | Drop: 0.2%


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]


===== finetune / train =====
ROWS: 57488
NULL VALUES:
None
ar => 44947 samples 78.2%
en => 12541 samples 21.8%

===== Text Length Distribution =====
count    114976.0
mean         92.1
std          53.8
min          10.0
25%          56.0
50%          77.0
75%         111.0
max         423.0
dtype: float64

===== Approximate Token Length =====
count    57488.0
mean        52.2
std         21.9
min          9.0
25%         35.0
50%         48.0
75%         69.0
max        197.0
dtype: float64

===== Context Window Analysis =====
Context Length: 512 | Keep: 100.0% | Drop: 0.0%
Context Length: 1024 | Keep: 100.0% | Drop: 0.0%
Context Length: 2048 | Keep: 100.0% | Drop: 0.0%


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]


===== finetune / eval =====
ROWS: 6379
NULL VALUES:
None
ar => 4976 samples 78.0%
en => 1403 samples 22.0%

===== Text Length Distribution =====
count    12758.0
mean        92.9
std         54.6
min         11.0
25%         56.0
50%         78.0
75%        113.0
max        515.0
dtype: float64

===== Approximate Token Length =====
count    6379.0
mean       52.7
std        21.9
min        10.0
25%        35.0
50%        49.0
75%        70.0
max       195.0
dtype: float64

===== Context Window Analysis =====
Context Length: 512 | Keep: 100.0% | Drop: 0.0%
Context Length: 1024 | Keep: 100.0% | Drop: 0.0%
Context Length: 2048 | Keep: 100.0% | Drop: 0.0%


In [47]:
df = load_dataset(
    REPO,
    "pretrain",
    split="phase1_train"
).to_pandas()

# Check duplicate text
text_dups = df.duplicated(
    subset=["text"]
).sum()

print(f"Rows with exact duplicate text: {text_dups}")

# Check duplicate article_id + chunk_index
pair_dups = df.duplicated(
    subset=["article_id", "chunk_index"]
).sum()

print(f"Duplicate (article_id, chunk_index) pairs: {pair_dups}")

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Rows with exact duplicate text: 0
Duplicate (article_id, chunk_index) pairs: 0
